# Lab 62: OCR-reading quality

Scoring read text is subtler than edit distance: CER/WER are too harsh on formatting and too lenient on the digits that carry the meaning. Build CER/WER, normalization, and a value-aware numeric match, and find where each is the right tool. Fill in the `TODO` cell; reference in `solution/`. Math: [math-foundations/17](../../math-foundations/17-multimodal-eval-metrics.md).

## Step 0: Setup and the full table

In [ ]:
from ocr_eval import cer, wer, normalize, extract_number, numeric_match, report
# CER and WER are edit distances - the standard OCR metrics. On numeric answers they mislead in
# both directions, so the answer span also needs a value-aware check.
for r in report():
    print(f"  {r['note']:28} CER {r['cer']:.2f}  CER(norm) {r['cer_norm']:.2f}  WER {r['wer']:.2f}  numeric {r['numeric']}")

## Step 1: CER vs WER

In [ ]:
# CER counts character edits; WER counts word edits. A dropped word out of three is WER 1/3.
print("WER, dropped one of three words:", wer("retriever reranker generator", "retriever generator"))
print("CER, one substituted char in three:", cer("abc", "abd"))

## Step 2: Normalization removes cosmetics, not content

In [ ]:
# Normalization removes cosmetic differences (case, whitespace) BEFORE scoring - it must never
# change content, only formatting.
ref, hyp = "4.2 million", "4.2  Million"
print(f"raw CER {cer(ref,hyp):.2f}  ->  normalized CER {cer(normalize(ref),normalize(hyp)):.2f}")

## Step 3: The misread-decimal trap (tiny CER, huge error)

In [ ]:
# TODO: take "4.2 million" misread as "42 million". Compute the normalized CER, then compare
# extract_number on each and numeric_match. Why does a tiny CER correspond to a catastrophic error,
# and what metric catches it?
raise NotImplementedError

## Step 4: Numeric tolerance (large CER, correct value)

In [ ]:
# The opposite case: a different FORMAT is a large CER but the same value. CER is too harsh here;
# numeric tolerance gets it right.
for ref,hyp in [("4.2 million","$4.2M"), ("63 percent","63%")]:
    print(f"{ref!r:14} vs {hyp!r:8}  CER {cer(normalize(ref),normalize(hyp)):.2f}  numeric_match {numeric_match(ref,hyp)}")

## What you built

The OCR-reading metric, with its trap. `cer`/`wer` are edit distances (the standard OCR/ASR metrics); `normalize` strips cosmetic differences before scoring; `extract_number`/`numeric_match` add a value-aware check for the answer span. The two lessons the numbers force: a **misread decimal** ('4.2'->'42') is only ~0.09 CER but a 10x value error - CER is too lenient on the characters that matter most - while a **format difference** ('$4.2M' vs '4.2 million') is ~0.7 CER but numerically identical - CER is too harsh on formatting. So report CER/WER for the read text overall *and* a numeric/structured match for the answer span; neither alone is enough.

**Where this simplifies:** the cases are hand-picked to expose the failure modes, and `extract_number` handles a small set of magnitudes and percent - a production parser needs currencies, dates, ranges, and units. The metric design is the point: edit distance for legibility, value tolerance for correctness. This is the OCR axis of [Lab 61](../61-grading-multimodal-rag/); the math is in [math-foundations/17](../../math-foundations/17-multimodal-eval-metrics.md).